In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import nltk
import re
import networkx as nx
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity

# Download necessary tokenizer models
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")  # FIX for sentence tokenization error

# File upload (for Google Colab or Jupyter)
from google.colab import files
uploaded = files.upload()  # Upload your .csv or .xls dataset manually

# Load the dataset with correct encoding
df = pd.read_csv("tennis_articles.csv", encoding="latin1")
df.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Saving tennis_articles.csv to tennis_articles (2).csv


,article_id,article_title,article_text,source
0,1,"I do not have friends in tennis, says Maria Sh...",Maria Sharapova has basically no friends as te...,https://www.tennisworldusa.org/tennis/news/Mar...
1,2,Federer defeats Medvedev to advance to 14th Sw...,"BASEL, Switzerland (AP)  Roger Federer advanc...",http://www.tennis.com/pro-game/2018/10/copil-s...
2,3,Tennis: Roger Federer ignored deadline set by ...,Roger Federer has revealed that organisers of ...,https://scroll.in/field/899938/tennis-roger-fe...
3,4,Nishikori to face off against Anderson in Vien...,Kei Nishikori will try to end his long losing ...,http://www.tennis.com/pro-game/2018/10/nishiko...
4,5,Roger Federer has made this huge change to ten...,"Federer, 37, first broke through on tour over ...",https://www.express.co.uk/sport/tennis/1036101...


In [9]:
# Inspect the structure of the dataset
df.info()
df.head()

# Drop title column if present
if "article_title" in df.columns:
    df = df.drop("article_title", axis=1)

# Extract the article text column
texts = df["article_text"].values


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   article_id     8 non-null      int64 
 1   article_title  8 non-null      object
 2   article_text   8 non-null      object
 3   source         8 non-null      object
dtypes: int64(1), object(3)
memory usage: 388.0+ bytes


In [10]:
# Tokenize all articles into sentences
sentences = []
for text in texts:
    sentences.extend(sent_tokenize(text))

print("Total number of sentences:", len(sentences))
sentences[:5]


Total number of sentences: 130


['Maria Sharapova has basically no friends as tennis players on the WTA Tour.',
 "The Russian player has no problems in openly speaking about it and in a recent interview she said: 'I don't really hide any feelings too much.",
 'I think everyone knows this is my job here.',
 "When I'm on the courts or when I'm on the court playing, I'm a competitor and I want to beat every single person whether they're in the locker room or across the net.",
 "So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match."]

In [15]:
# Auto-check and download GloVe if needed
import os

if "glove.6B.100d.txt" not in os.listdir():
    print("GloVe file not found → downloading it...")
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip glove.6B.zip
else:
    print("GloVe file found!")

# Load GloVe embeddings
glove_embeddings = {}

with open("glove.6B.100d.txt", "r", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

print("Loaded word vectors:", len(glove_embeddings))


GloVe file found!
Loaded word vectors: 400000


In [16]:
# Define English stopwords
stop_words = set(stopwords.words("english"))

def clean_sentence(sentence):
    # Lowercase
    sentence = sentence.lower()
    # Remove numbers, punctuation, special characters
    sentence = re.sub(r"[^a-zA-Z]", " ", sentence)
    # Remove extra spaces
    sentence = re.sub(r"\s+", " ", sentence).strip()
    # Remove stopwords
    words = [w for w in sentence.split() if w not in stop_words]
    return " ".join(words)

# Apply cleaning
cleaned_sentences = [clean_sentence(s) for s in sentences]

cleaned_sentences[:5]


['maria sharapova basically friends tennis players wta tour',
 'russian player problems openly speaking recent interview said really hide feelings much',
 'think everyone knows job',
 'courts court playing competitor want beat every single person whether locker room across net',
 'one strike conversation weather know next minutes go try win tennis match']

In [17]:
# Convert cleaned sentences into vectors by averaging word embeddings
def sentence_vector(sentence):
    words = sentence.split()
    if len(words) == 0:
        return np.zeros(100)

    vectors = []
    for w in words:
        if w in glove_embeddings:
            vectors.append(glove_embeddings[w])
        else:
            vectors.append(np.zeros(100))  # For unknown words

    return np.mean(vectors, axis=0)

sentence_vectors = [sentence_vector(s) for s in cleaned_sentences]

len(sentence_vectors)


130

In [18]:
# Create similarity matrix
n = len(sentence_vectors)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            similarity_matrix[i][j] = cosine_similarity(
                sentence_vectors[i].reshape(1, 100),
                sentence_vectors[j].reshape(1, 100)
            )[0, 0]


In [19]:
# Create graph from similarity matrix
graph = nx.from_numpy_array(similarity_matrix)

# Apply PageRank algorithm
scores = nx.pagerank(graph)

# Rank sentences by their PageRank score
ranked_sentences = sorted(
    ((scores[i], s) for i, s in enumerate(sentences)),
    reverse=True
)


In [20]:
# Number of sentences in the final summary
N = 10

summary = [ranked_sentences[i][1] for i in range(N)]

print("===== SUMMARY =====")
for s in summary:
    print("-", s)


===== SUMMARY =====
- I was on a nice trajectorythen, Reid recalled.If I hadnt got sick, I think I could have started pushing towards the second week at the slams and then who knows. Duringa comeback attempt some five years later, Reid added Bernard Tomic and 2018 US Open Federer slayer John Millman to his list of career scalps.
- Major players feel that a big event in late November combined with one in January before the Australian Open will mean too much tennis and too little rest.
- So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match.
- Speaking at the Swiss Indoors tournament where he will play in Sundays final against Romanian qualifier Marius Copil, the world number three said that given the impossibly short time frame to make a decision, he opted out of any commitment.
- Currently in ninth place, Nishikori with a win could move to within 125 points of the cut for the eight-man eve